In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd() if (Path.cwd() / "open_agent_compiler").exists() else Path.cwd().parent
sys.path.insert(0, str(project_root))

# Test agent definition

In [3]:
from pathlib import Path

from open_agent_compiler.model.core.agent_model import AgentHeader, AgentDefinition, AgentVariant, ModelParameters
from open_agent_compiler.model.core.skills_model import SkillDefinition, WorkflowStep, SkillExample
from open_agent_compiler.model.core.tools_model import (
    ToolDefinition, ToolDefinitionHeader, ToolDefinitionLogicBash,
    ToolDefinitionLogicJson, ToolScriptDefinition, ScriptDefinition,
)
from open_agent_compiler.model.core.permissions_model import JsonToolPermission, BashToolPermission

# ---------------------------------------------------------------------------
# ScriptTool Introspection
# ---------------------------------------------------------------------------
import importlib.util
from open_agent_compiler.runtime import ScriptTool

project_root = Path.cwd() if (Path.cwd() / "open_agent_compiler").exists() else Path.cwd().parent
hello_spec = importlib.util.spec_from_file_location("hello_mod", str(project_root / "scripts/example/hello.py"))
hello_mod = importlib.util.module_from_spec(hello_spec)
hello_spec.loader.exec_module(hello_mod)
Hello = hello_mod.Hello

random_word_spec = importlib.util.spec_from_file_location("random_word_mod", str(project_root / "scripts/example/random_word.py"))
random_word_mod = importlib.util.module_from_spec(random_word_spec)
random_word_spec.loader.exec_module(random_word_mod)
RandomWord = random_word_mod.RandomWord

print("=== ScriptTool Introspection ===")
print(f"Hello: name={Hello.name}, desc={Hello.description}, schema={Hello.get_input_schema()}")
print(f"RandomWord: name={RandomWord.name}, desc={RandomWord.description}, schema={RandomWord.get_input_schema()}")

# ---------------------------------------------------------------------------
# TypeScript Template
# ---------------------------------------------------------------------------
ts_template = (project_root / "open_agent_compiler/compiler/templates/tool.ts").read_text()
print("\n=== TypeScript Tool Template ===")
print(ts_template)

# ---------------------------------------------------------------------------
# Generated TypeScript examples
# ---------------------------------------------------------------------------
print("\n=== Generated: .opencode/tools/hello.ts ===")
print('''import { tool } from "@opencode-ai/plugin"
import path from "path"

export default tool({
  description: "Prints 'hello' and returns it as a string.",
  args: {},
  async execute(args, context) {
    const script = path.join(context.worktree, "scripts/example/hello.py")
    const result = await Bun.stdin(JSON.stringify(args)).$`uv run ${script} --json`.text()
    return result.trim()
  },
})
''')

print("\n=== Generated: .opencode/tools/random_word.ts ===")
print('''import { tool } from "@opencode-ai/plugin"
import path from "path"

export default tool({
  description: "Returns a random word from a fixed list of 10 words.",
  args: {},
  async execute(args, context) {
    const script = path.join(context.worktree, "scripts/example/random_word.py")
    const result = await Bun.stdin(JSON.stringify(args)).$`uv run ${script} --json`.text()
    return result.trim()
  },
})
''')

# ---------------------------------------------------------------------------
# Tool Definitions (both bash and json variants)
# ---------------------------------------------------------------------------
hello_tool = ToolDefinition(
    header=ToolDefinitionHeader(
        name="hello",
        description="Prints 'hello' and returns it as a string.",
        usage_explanation_long="Runs scripts/example/hello.py which uses the ScriptTool interface.",
        usage_explanation_short="Prints and returns 'hello'.",
        rules=["Always return the exact string 'hello'."],
    ),
    bash_tool=ToolDefinitionLogicBash(
        permission_bash=BashToolPermission(tool_name="bash", value="allow"),
        positive_examples=["uv run scripts/example/hello.py"],
        negative_examples=["Use python3 instead of uv run"],
        mode_specific_rules=["Always use uv run"],
    ),
    json_tool=ToolDefinitionLogicJson(
        permission_json=JsonToolPermission(tool_name="custom_tool", value="allow"),
        positive_examples=["hello()"],
        negative_examples=["Call via bash directly"],
        mode_specific_rules=["Use typed tool call"],
        tool_scripts=[ToolScriptDefinition(
            paths=[project_root / "scripts/example/hello.py"],
            scripts=[ScriptDefinition(
                target_file_path=project_root / "scripts/example/hello.py",
                source_file_path=project_root / "scripts/example/hello.py",
                source_file_type="python",
                script_contents=None,
            )],
        )],
    ),
)

random_word_tool = ToolDefinition(
    header=ToolDefinitionHeader(
        name="random_word",
        description="Returns a random word from a fixed list of 10 words.",
        usage_explanation_long="Runs scripts/example/random_word.py which returns a random word as JSON.",
        usage_explanation_short="Returns a random word.",
        rules=["Always return the exact word from the script."],
    ),
    bash_tool=ToolDefinitionLogicBash(
        permission_bash=BashToolPermission(tool_name="bash", value="allow"),
        positive_examples=["uv run scripts/example/random_word.py"],
        negative_examples=["Use python3 instead of uv run"],
        mode_specific_rules=["Always use uv run"],
    ),
    json_tool=ToolDefinitionLogicJson(
        permission_json=JsonToolPermission(tool_name="custom_tool", value="allow"),
        positive_examples=["random_word()"],
        negative_examples=["Call via bash directly"],
        mode_specific_rules=["Use typed tool call"],
        tool_scripts=[ToolScriptDefinition(
            paths=[project_root / "scripts/example/random_word.py"],
            scripts=[ScriptDefinition(
                target_file_path=project_root / "scripts/example/random_word.py",
                source_file_path=project_root / "scripts/example/random_word.py",
                source_file_type="python",
                script_contents=None,
            )],
        )],
    ),
)

print("\n=== Tool Definitions ===")
print(f"hello: bash={hello_tool.bash_tool is not None}, json={hello_tool.json_tool is not None}")
print(f"random_word: bash={random_word_tool.bash_tool is not None}, json={random_word_tool.json_tool is not None}")

# ---------------------------------------------------------------------------
# Skill Definitions
# ---------------------------------------------------------------------------
greeting_skill = SkillDefinition(
    name="greeting",
    description="Greets the user using the hello tool.",
    usage_explanation_long="Uses the hello tool to produce a greeting message.",
    usage_explanation_short="Greets the user.",
    rules=["Use the hello tool to greet."],
    workflow_steps=[
        WorkflowStep(
            header="Greet",
            condition="User asks for a greeting.",
            result="Returns 'hello' string.",
            rule="Invoke the hello tool.",
            tools_used=[hello_tool],
        )
    ],
    positive_examples=[
        SkillExample(
            header="Standard greeting",
            condition="User says hi.",
            result="Returns 'hello'.",
            rule="Call hello tool.",
            tools_used=[hello_tool],
        )
    ],
    negative_examples=[],
)

joke_skill = SkillDefinition(
    name="joke_writer",
    description="Generates a witty joke based on a randomly selected word.",
    usage_explanation_long="First calls the random_word tool to get a random word, then writes a short funny joke incorporating that word.",
    usage_explanation_short="Writes a joke based on a random word.",
    rules=[
        "Always call the random_word tool first.",
        "Write a short, witty joke incorporating the returned word.",
    ],
    workflow_steps=[
        WorkflowStep(
            header="Get word",
            condition="Always.",
            result="Random word from the tool.",
            rule="Call random_word tool.",
            tools_used=[random_word_tool],
        ),
        WorkflowStep(
            header="Write joke",
            condition="After getting the word.",
            result="A funny joke using the word.",
            rule="Write a clean, family-friendly joke.",
            tools_used=[],
        )
    ],
    positive_examples=[
        SkillExample(
            header="Joke about 'apple'",
            condition="random_word returns 'apple'.",
            result="A short joke about apples.",
            rule="Call random_word, then write joke.",
            tools_used=[random_word_tool],
        )
    ],
    negative_examples=[],
)

# ---------------------------------------------------------------------------
# Agent Definitions
# ---------------------------------------------------------------------------
joke_subagent = AgentDefinition(
    header=AgentHeader(
        agent_id="joke_subagent",
        name="Joke Subagent",
        description="Generates a joke based on a random word.",
    ),
    skills=[joke_skill],
    subagents=[],
    extra_tools=[random_word_tool],
    usage_explanation_long="Subagent that generates jokes using the random_word tool.",
    usage_explanation_short="Generates jokes.",
)

hello_agent = AgentDefinition(
    header=AgentHeader(
        agent_id="hello_agent",
        name="Hello Agent",
        description="Primary agent that greets users and can delegate joke generation.",
    ),
    skills=[greeting_skill],
    subagents=[joke_subagent.header],
    extra_tools=[hello_tool],
    usage_explanation_long="Primary agent that greets users via the hello tool and can delegate to the joke subagent.",
    usage_explanation_short="Greets and tells jokes.",
)

# ---------------------------------------------------------------------------
# Variant
# ---------------------------------------------------------------------------
agent_variant = AgentVariant(
    postfix="v1",
    agent_mode="primary",
    agent_definition=hello_agent,
    model_parameters=ModelParameters(model_name="gpt-4o", temperature=0.7),
)

print("\n=== Agent Structure ===")
print(f"Primary: {agent_variant.agent_definition.header.name}")
print(f"  skills: {[s.name for s in agent_variant.agent_definition.skills]}")
print(f"  subagents: {[a.name for a in agent_variant.agent_definition.subagents]}")
print(f"  extra_tools: {[t.header.name for t in agent_variant.agent_definition.extra_tools]}")
print(f"  model: {agent_variant.model_parameters.model_name}")
print("\nDone!")


=== ScriptTool Introspection ===
Hello: name=hello, desc=Prints 'hello' and returns it as a string, schema=[]
RandomWord: name=random_word, desc=Returns a random word from a fixed list of 10 words, schema=[]

=== TypeScript Tool Template ===
import { tool } from "@opencode-ai/plugin"
import path from "path"

export default tool({
  description: "{{description}}",
  args: {
{{args}}
  },
  async execute(args, context) {
    const script = path.join(context.worktree, "{{script_path}}")
    const result = await Bun.stdin(JSON.stringify(args)).$`uv run ${script} --json`.text()
    return result.trim()
  },
})


=== Generated: .opencode/tools/hello.ts ===
import { tool } from "@opencode-ai/plugin"
import path from "path"

export default tool({
  description: "Prints 'hello' and returns it as a string.",
  args: {},
  async execute(args, context) {
    const script = path.join(context.worktree, "scripts/example/hello.py")
    const result = await Bun.stdin(JSON.stringify(args)).$`uv run ${sc

In [4]:
# Future: add compilation logic here